# FL-06 GitHub-only source-row audit — v2 reproducibility

**Purpose:** independently reproduce the exact v1.3 group-aware split and compare reconstructed outputs against archived v1 from the eight checksum-verified CIC-IDS2017 source CSVs; verify every Train/Validation mixed-label feature-hash group against all 77 retained feature values; and trace negative values in nonnegative-semantic features to source file and row.

**Read-only safety:** this notebook does not train or predict, does not load the sealed Test arrays or perform Test-specific error/conflict analysis; it reads the raw source rows only as required to reproduce the already-fixed group split and its fingerprint, and does not compute Test metrics, does not modify labels or source CSVs, and does not alter the final-model lock, Test record, checkpoints, or existing experiment outputs. It writes new audit-only CSV/JSON/Markdown files into `audit/source_row_audit_v2_reproducibility`; archived v1 outputs are read-only and never overwritten.

**Run:** open in Colab and run the notebook through the v1-v2 comparison cell. Run the final GitHub publication cell only if you want the generated audit outputs committed back to the repository. Add `GITHUB_TOKEN` to Colab Secrets and enable notebook access. Read access is enough to run the audit; the final optional publish cell requires Contents: Read and Write. You may skip that final cell and send the generated files instead. The token is read from Secrets and never printed or saved. The notebook may require substantial RAM because it reproduces the original full-data v1.3 preparation exactly. If it runs out of RAM, send the full error; do not change the cleaning or split logic manually.

In [3]:
import os, sys, json, csv, hashlib, base64, subprocess, shutil, gc, platform
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

REPO = "AmirHosseinSharifian/Federated-Learning"
ROOT = Path("/content/Federated-Learning-audit-v2")
PROJECT = ROOT / "CIC-IDS2017"
DATA_DIR = PROJECT / "data"
FRAMEWORK_ROOT = PROJECT / "FL-06_OneShot_GroupAware_v1_3_FinalPaper"
AUDIT_DIR = FRAMEWORK_ROOT / "audit"
OUT_DIR = AUDIT_DIR / "source_row_audit_v2_reproducibility"
MANIFEST = AUDIT_DIR / "raw_file_manifest.csv"
SPLIT_MANIFEST = AUDIT_DIR / "dataset_split_manifest.json"
GROUP_AUDIT = AUDIT_DIR / "feature_group_split_audit.json"

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets and enable notebook access.") from exc
if not TOKEN:
    raise RuntimeError("GITHUB_TOKEN is missing from Colab Secrets.")

auth = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
AUTH_CONFIG = f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}"
ENV = os.environ.copy()
ENV["GIT_LFS_SKIP_SMUDGE"] = "1"
ENV["GIT_TERMINAL_PROMPT"] = "0"

def git(args, cwd=None, check=True):
    p = subprocess.run(["git", "-c", AUTH_CONFIG, *args], cwd=cwd, env=ENV,
                       text=True, capture_output=True)
    if check and p.returncode:
        msg = (p.stderr or p.stdout).replace(TOKEN, "[REDACTED]")
        raise RuntimeError(f"Git failed ({p.returncode}): {msg[-4000:]}")
    return p

print("Python:", platform.python_version(), "| pandas:", pd.__version__, "| NumPy:", np.__version__)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())
if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
subprocess.run(["git", "lfs", "install", "--system"], check=True)

if ROOT.exists():
    shutil.rmtree(ROOT)
git(["clone", "--depth", "1", f"https://github.com/{REPO}.git", str(ROOT)])
git(["lfs", "install", "--local"], cwd=ROOT)
git(["lfs", "pull", "--include=CIC-IDS2017/data/*.csv"], cwd=ROOT)
print("Repository HEAD:", git(["rev-parse", "HEAD"], cwd=ROOT).stdout.strip())
print("Git LFS pull completed.")


Python: 3.13.18 | pandas: 2.2.3 | NumPy: 2.4.6
Git: git version 2.43.0
Repository HEAD: c5129c433862f519b9a1dd306bff25b2af8a2c54
Git LFS pull completed.


In [4]:
# Verify all eight source files before reading any rows.
if not MANIFEST.is_file():
    raise FileNotFoundError(MANIFEST)
with MANIFEST.open("r", encoding="utf-8-sig", newline="") as f:
    manifest_rows = list(csv.DictReader(f))
if len(manifest_rows) != 8:
    raise RuntimeError(f"Expected 8 source files, found {len(manifest_rows)}")

verified_files = []
source_verification_rows = []
for rec in manifest_rows:
    name = rec["file"]
    expected_size = int(rec["size_bytes"])
    expected_sha = rec["sha256"].strip().lower()
    p = DATA_DIR / name
    if not p.is_file():
        raise FileNotFoundError(p)
    h = hashlib.sha256()
    size = 0
    with p.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            size += len(chunk)
            h.update(chunk)
    actual_sha = h.hexdigest()
    if size != expected_size or actual_sha != expected_sha:
        raise RuntimeError(f"Integrity failure: {name}; size {size}/{expected_size}; SHA {actual_sha}/{expected_sha}")
    verified_files.append(name)
    source_verification_rows.append({"file": name, "expected_size_bytes": expected_size, "actual_size_bytes": size, "size_match": size == expected_size, "expected_sha256": expected_sha, "actual_sha256": actual_sha, "sha256_match": actual_sha == expected_sha, "status": "PASS"})
    print(f"PASS {name}: {size:,} bytes; SHA-256 exact")

expected_split = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
expected_groups = json.loads(GROUP_AUDIT.read_text(encoding="utf-8"))
print("\nExpected dataset fingerprint:", expected_split["dataset_fingerprint"])
print("Expected split fingerprint:", expected_split["split_fingerprint"])
print("Expected group split trial:", expected_groups["selected_trial_index"])
print("Expected conflict audit: 614 groups / 6,342 Train+Validation rows")


PASS Monday-WorkingHours.pcap_ISCX.csv: 176,927,918 bytes; SHA-256 exact
PASS Tuesday-WorkingHours.pcap_ISCX.csv: 135,078,995 bytes; SHA-256 exact
PASS Wednesday-workingHours.pcap_ISCX.csv: 225,166,395 bytes; SHA-256 exact
PASS Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 52,023,263 bytes; SHA-256 exact
PASS Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 83,102,436 bytes; SHA-256 exact
PASS Friday-WorkingHours-Morning.pcap_ISCX.csv: 58,316,725 bytes; SHA-256 exact
PASS Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 77,123,859 bytes; SHA-256 exact
PASS Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 76,906,168 bytes; SHA-256 exact

Expected dataset fingerprint: 831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58
Expected split fingerprint: b67f17d162df41840629b22fa258b7e843cf54541ccfd6fff5910f325b70b812
Expected group split trial: 40
Expected conflict audit: 614 groups / 6,342 Train+Validation rows


In [5]:
# Reconstruct the same cleaned 77-feature table and v1.3 group split.
# This intentionally mirrors the v1.3 Runner's preprocessing/split code.
FILES = [r["file"] for r in manifest_rows]
frames, file_manifest = [], []
raw_offset = 0

for file_number, filename in enumerate(FILES, start=1):
    target = DATA_DIR / filename
    print(f"[{file_number}/8] Reading {filename} ({target.stat().st_size / (1024**2):.1f} MiB)")
    frame = pd.read_csv(target, low_memory=False)
    frame.columns = frame.columns.astype(str).str.strip()
    if frame.columns.duplicated().any():
        raise ValueError(f"Duplicate columns after trimming in {filename}")
    if "Label" not in frame.columns:
        raise ValueError(f"Label column missing in {filename}")
    frame["Label"] = frame["Label"].astype("string").str.strip()
    # Match the original manifest convention: source-file column is included in columns_loaded.
    frame["_source_file"] = filename
    file_manifest.append({
        "file": filename, "size_bytes": int(target.stat().st_size),
        "sha256": next(x["sha256"] for x in manifest_rows if x["file"] == filename),
        "rows_loaded": int(len(frame)), "columns_loaded": int(len(frame.columns))
    })
    frame["_source_row_index"] = np.arange(len(frame), dtype=np.int64)
    frame["_raw_global_row_index"] = np.arange(raw_offset, raw_offset + len(frame), dtype=np.int64)
    raw_offset += len(frame)
    frames.append(frame)

raw_df = pd.concat(frames, ignore_index=True)
del frames
gc.collect()
raw_rows = int(len(raw_df))
raw_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Exactly the 78 numeric source features, excluding metadata and target.
excluded = {"Label", "_source_file", "_source_row_index", "_raw_global_row_index"}
source_feature_columns = [c for c in raw_df.columns if c not in excluded]
if len(source_feature_columns) != 78:
    raise RuntimeError(f"Expected 78 source numeric columns, got {len(source_feature_columns)}")
raw_labels = raw_df["Label"].copy()
raw_df[source_feature_columns] = raw_df[source_feature_columns].apply(pd.to_numeric, errors="coerce")
valid_label = raw_labels.notna() & raw_labels.astype(str).str.strip().ne("")
valid_feature = raw_df[source_feature_columns].notna().all(axis=1)
valid_mask = valid_label & valid_feature
clean_df = raw_df.loc[valid_mask].copy().reset_index(drop=True)
clean_df["Label"] = clean_df["Label"].astype(str).str.strip()
rows_removed = raw_rows - len(clean_df)

# v1.3 removes only the verified duplicate feature column; raw source data remain untouched.
if "Fwd Header Length" not in source_feature_columns or "Fwd Header Length.1" not in source_feature_columns:
    raise RuntimeError("Expected duplicated header-length columns are missing.")
if not np.array_equal(clean_df["Fwd Header Length"].to_numpy(),
                      clean_df["Fwd Header Length.1"].to_numpy()):
    raise RuntimeError("Fwd Header Length duplicate check failed; stop without changing data.")
FEATURE_COLUMNS = [c for c in source_feature_columns if c != "Fwd Header Length.1"]
if len(FEATURE_COLUMNS) != 77:
    raise RuntimeError(f"Expected 77 model features, got {len(FEATURE_COLUMNS)}")

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(clean_df["Label"].astype(str)).astype(np.int32)
CLASS_NAMES = [str(x) for x in label_encoder.classes_]
NUM_CLASSES = len(CLASS_NAMES)
if NUM_CLASSES != 15:
    raise RuntimeError(f"Expected 15 classes, found {NUM_CLASSES}: {CLASS_NAMES}")

# Preserve the exact source provenance for every clean row.
source_file_all = clean_df["_source_file"].astype(str).to_numpy()
source_row_all = clean_df["_source_row_index"].to_numpy(dtype=np.int64)
raw_global_all = clean_df["_raw_global_row_index"].to_numpy(dtype=np.int64)
feature_hashes = pd.util.hash_pandas_object(clean_df[FEATURE_COLUMNS], index=False).to_numpy(np.uint64)
all_indices = np.arange(len(y_all), dtype=np.int64)

print("\nRaw rows:", raw_rows, "| removed:", rows_removed, "| clean:", len(clean_df))
print("Model features:", len(FEATURE_COLUMNS), "| classes:", NUM_CLASSES)
print("Class order:", CLASS_NAMES)


[1/8] Reading Monday-WorkingHours.pcap_ISCX.csv (168.7 MiB)
[2/8] Reading Tuesday-WorkingHours.pcap_ISCX.csv (128.8 MiB)
[3/8] Reading Wednesday-workingHours.pcap_ISCX.csv (214.7 MiB)
[4/8] Reading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv (49.6 MiB)
[5/8] Reading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv (79.3 MiB)
[6/8] Reading Friday-WorkingHours-Morning.pcap_ISCX.csv (55.6 MiB)
[7/8] Reading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (73.6 MiB)
[8/8] Reading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (73.3 MiB)

Raw rows: 2830743 | removed: 2867 | clean: 2827876
Model features: 77 | classes: 15
Class order: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']


In [6]:
# Reproduce the deterministic 50-trial group-level split and verify its fingerprints.
_group_codes, _unique_feature_hashes = pd.factorize(feature_hashes, sort=False)
if np.any(_group_codes < 0):
    raise RuntimeError("Invalid feature-hash group code.")
_n_groups = int(len(_unique_feature_hashes))
_overall_class_counts = np.bincount(y_all, minlength=NUM_CLASSES).astype(np.int64)
_target_fractions = np.asarray([0.80, 0.10, 0.10], dtype=np.float64)
_best_group_partition = None
_best_group_score = np.inf
_valid_group_trials = 0
GROUP_SPLIT_TRIALS = 50

for trial in range(GROUP_SPLIT_TRIALS):
    rng = np.random.default_rng(42 + 1000 + trial)
    u = rng.random(_n_groups)
    group_assignment = np.zeros(_n_groups, dtype=np.int8)
    group_assignment[u < 0.10] = 2
    group_assignment[(u >= 0.10) & (u < 0.20)] = 1
    row_assignment = group_assignment[_group_codes]
    candidate_indices = [np.flatnonzero(row_assignment == split_id).astype(np.int64)
                         for split_id in range(3)]
    support = np.vstack([np.bincount(y_all[ids], minlength=NUM_CLASSES)
                         for ids in candidate_indices])
    if np.any(support == 0):
        continue
    _valid_group_trials += 1
    expected_support = _target_fractions[:, None] * _overall_class_counts[None, :]
    class_error = float(np.mean(np.abs(support - expected_support) /
                                np.maximum(expected_support, 1.0)))
    row_fractions = np.asarray([len(ids) / len(y_all) for ids in candidate_indices], dtype=np.float64)
    row_error = float(np.mean(np.abs(row_fractions - _target_fractions)))
    score = class_error + 2.0 * row_error
    if score < _best_group_score:
        _best_group_score = score
        _best_group_partition = (candidate_indices[0].copy(), candidate_indices[1].copy(),
                                 candidate_indices[2].copy(), group_assignment.copy(),
                                 support.copy(), row_fractions.copy(), int(trial))

if _best_group_partition is None:
    raise RuntimeError("No valid group-aware partition found.")
idx_train, idx_val, idx_test, best_group_assignment, best_support, best_fractions, selected_trial = _best_group_partition
if selected_trial != int(expected_groups["selected_trial_index"]):
    raise RuntimeError(f"Selected split trial mismatch: {selected_trial} vs {expected_groups['selected_trial_index']}")

def indices_fingerprint(*arrays):
    digest = hashlib.sha256()
    for arr in arrays:
        arr = np.ascontiguousarray(arr, dtype=np.int64)
        digest.update(np.asarray(arr.shape, dtype=np.int64).view(np.uint8))
        digest.update(arr.view(np.uint8))
    return digest.hexdigest()

data_digest = hashlib.sha256()
data_digest.update(json.dumps(file_manifest, sort_keys=True).encode("utf-8"))
data_digest.update(np.ascontiguousarray(feature_hashes).view(np.uint8))
data_digest.update(np.ascontiguousarray(y_all).view(np.uint8))
data_digest.update(json.dumps(FEATURE_COLUMNS, ensure_ascii=False).encode("utf-8"))
DATASET_FINGERPRINT = data_digest.hexdigest()
SPLIT_FINGERPRINT = indices_fingerprint(idx_train, idx_val, idx_test)

print("Selected group-split trial:", selected_trial)
print("Valid trials:", _valid_group_trials, "/", GROUP_SPLIT_TRIALS)
print("Dataset fingerprint:", DATASET_FINGERPRINT)
print("Split fingerprint:", SPLIT_FINGERPRINT)
print("Expected dataset fingerprint:", expected_split["dataset_fingerprint"])
print("Expected split fingerprint:", expected_split["split_fingerprint"])

if DATASET_FINGERPRINT != expected_split["dataset_fingerprint"]:
    raise RuntimeError("DATASET FINGERPRINT MISMATCH. Stop; do not interpret split assignments.")
if SPLIT_FINGERPRINT != expected_split["split_fingerprint"]:
    raise RuntimeError("SPLIT FINGERPRINT MISMATCH. Stop; do not interpret split assignments.")
if len(idx_train) != expected_split["train_rows"] or len(idx_val) != expected_split["validation_rows"] or len(idx_test) != expected_split["test_rows_sealed"]:
    raise RuntimeError("Split row counts do not match the committed manifest.")

# Confirm that each feature-hash group belongs to exactly one split.
split_by_clean_row = np.full(len(y_all), -1, dtype=np.int8)
split_by_clean_row[idx_train] = 0
split_by_clean_row[idx_val] = 1
split_by_clean_row[idx_test] = 2
if np.any(split_by_clean_row < 0):
    raise AssertionError("Some clean rows have no split assignment.")
print("Exact v1.3 dataset and split fingerprints verified.")


Selected group-split trial: 40
Valid trials: 25 / 50
Dataset fingerprint: 831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58
Split fingerprint: b67f17d162df41840629b22fa258b7e843cf54541ccfd6fff5910f325b70b812
Expected dataset fingerprint: 831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58
Expected split fingerprint: b67f17d162df41840629b22fa258b7e843cf54541ccfd6fff5910f325b70b812
Exact v1.3 dataset and split fingerprints verified.


In [7]:
# Source-row conflict audit: Train + Validation only. Test rows are not exported or inspected here.
train_val_idx = np.concatenate([idx_train, idx_val])
tv = pd.DataFrame({
    "clean_row_index": train_val_idx,
    "feature_hash": feature_hashes[train_val_idx].astype(np.uint64),
    "label": clean_df.loc[train_val_idx, "Label"].to_numpy(),
    "class_id": y_all[train_val_idx],
    "split": np.where(np.isin(train_val_idx, idx_train), "train", "validation")
})
group_stats = tv.groupby("feature_hash", sort=False).agg(
    rows=("label", "size"), unique_labels=("label", "nunique"),
    labels=("label", lambda s: " || ".join(sorted(set(map(str, s)))))
).reset_index()
conflict_stats = group_stats[group_stats["unique_labels"] > 1].copy()
conflict_hashes = set(conflict_stats["feature_hash"].astype("uint64").tolist())

if len(conflict_stats) != 614 or int(conflict_stats["rows"].sum()) != 6342:
    raise RuntimeError(
        f"Conflict reconstruction mismatch: {len(conflict_stats)} groups / "
        f"{int(conflict_stats['rows'].sum())} rows; expected 614 / 6342. "
        "No audit outputs will be treated as verified."
    )

# Exact vector comparison across all 77 columns, not only the 64-bit hash.
conflict_clean_idx = np.flatnonzero(np.isin(feature_hashes, np.fromiter(conflict_hashes, dtype=np.uint64)))
conflict_frame = clean_df.iloc[conflict_clean_idx].copy()
conflict_frame.insert(0, "clean_row_index", conflict_clean_idx)
conflict_frame.insert(1, "feature_hash", feature_hashes[conflict_clean_idx].astype(np.uint64))
conflict_frame.insert(2, "class_id", y_all[conflict_clean_idx])
conflict_frame.insert(3, "split", split_by_clean_row[conflict_clean_idx])
conflict_frame.rename(columns={
    "_source_file": "source_file",
    "_source_row_index": "source_file_row_index_zero_based",
    "_raw_global_row_index": "raw_global_row_index_zero_based"
}, inplace=True)

# Count exact distinct 77-feature vectors inside each conflicting hash group.
exact_rows = conflict_frame[["feature_hash", *FEATURE_COLUMNS]].drop_duplicates()
exact_counts = exact_rows.groupby("feature_hash", sort=False).size().rename("exact_77_feature_vectors")
conflict_stats["exact_77_feature_vectors"] = conflict_stats["feature_hash"].map(exact_counts).astype(int)
conflict_stats["hash_collision_detected"] = conflict_stats["exact_77_feature_vectors"] > 1
conflict_stats["split"] = conflict_stats["feature_hash"].map(
    conflict_frame.groupby("feature_hash")["split"].first()
)
conflict_stats["labels"] = conflict_stats["labels"].astype(str)

# Add provenance-rich rows, including all 77 retained model features.
metadata_cols = ["clean_row_index", "feature_hash", "class_id", "split",
                 "source_file", "source_file_row_index_zero_based",
                 "raw_global_row_index_zero_based", "Label"]
record_cols = [c for c in metadata_cols if c in conflict_frame.columns] + FEATURE_COLUMNS
conflict_records = conflict_frame[record_cols].copy()

# Check that the current v1.3 largest group is present and inspect it by exact vector.
largest_hash = int(conflict_stats.loc[conflict_stats["rows"].idxmax(), "feature_hash"])
largest_group = conflict_records[conflict_records["feature_hash"].astype("uint64") == np.uint64(largest_hash)].copy()
largest_exact_vectors = int(largest_group[FEATURE_COLUMNS].drop_duplicates().shape[0])
print("Reconstructed mixed-label groups:", len(conflict_stats))
print("Rows in those groups:", int(conflict_stats["rows"].sum()))
print("Largest group rows:", len(largest_group))
print("Largest group exact distinct 77-feature vectors:", largest_exact_vectors)
print("Largest group labels:", sorted(largest_group["Label"].astype(str).unique().tolist()))
print("Largest group source files:", largest_group["source_file"].value_counts().to_dict())

# Negative values in selected features whose CICFlowMeter semantics are nonnegative.
NONNEGATIVE_FEATURES = [
    "Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Max", "Fwd Packet Length Min", "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean", "Bwd Packet Length Std",
    "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
    "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std",
    "Packet Length Variance", "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
    "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
    "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes",
    "act_data_pkt_fwd", "min_seg_size_forward", "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min"
]
NONNEGATIVE_FEATURES = [c for c in NONNEGATIVE_FEATURES if c in FEATURE_COLUMNS]
tv_set = set(train_val_idx.tolist())
negative_parts = []
for feature in NONNEGATIVE_FEATURES:
    values = clean_df[feature].to_numpy()
    neg_idx = np.flatnonzero(values < 0)
    neg_idx = np.intersect1d(neg_idx, train_val_idx, assume_unique=False)
    if len(neg_idx):
        part = pd.DataFrame({
            "feature": feature,
            "value": values[neg_idx],
            "clean_row_index": neg_idx,
            "source_file": source_file_all[neg_idx],
            "source_file_row_index_zero_based": source_row_all[neg_idx],
            "raw_global_row_index_zero_based": raw_global_all[neg_idx],
            "Label": clean_df.loc[neg_idx, "Label"].to_numpy(),
            "split": np.where(np.isin(neg_idx, idx_train), "train", "validation")
        })
        negative_parts.append(part)

negative_records = pd.concat(negative_parts, ignore_index=True) if negative_parts else pd.DataFrame()
negative_feature_value_occurrences = int(len(negative_records))
negative_unique_source_rows = int(negative_records[["source_file", "source_file_row_index_zero_based"]].drop_duplicates().shape[0]) if len(negative_records) else 0
negative_summary = (negative_records.groupby(["feature", "source_file", "Label", "split"], dropna=False)
                    .size().rename("negative_rows").reset_index()) if len(negative_records) else pd.DataFrame()

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(source_verification_rows).to_csv(OUT_DIR / "source_file_verification.csv", index=False)
conflict_stats.sort_values(["rows", "feature_hash"], ascending=[False, True]).to_csv(
    OUT_DIR / "source_conflict_groups_train_validation.csv", index=False)
conflict_records.to_csv(OUT_DIR / "source_conflict_records_77_features_train_validation.csv", index=False)
negative_summary.to_csv(OUT_DIR / "negative_values_summary_train_validation.csv", index=False)
negative_records.to_csv(OUT_DIR / "negative_values_records_train_validation.csv", index=False)

summary = {
    "status": "completed",
    "scope": "Conflict and negative-value analysis restricted to Train and Validation; raw source rows are loaded to reproduce the fixed split, but no Test-specific analysis, export, prediction, metric, or tuning is performed",
    "repository_head": git(["rev-parse", "HEAD"], cwd=ROOT).stdout.strip(),
    "verified_source_files": len(verified_files),
    "raw_rows": int(raw_rows), "rows_removed": int(rows_removed), "clean_rows": int(len(clean_df)),
    "feature_count": int(len(FEATURE_COLUMNS)), "class_count": int(NUM_CLASSES),
    "dataset_fingerprint": DATASET_FINGERPRINT, "split_fingerprint": SPLIT_FINGERPRINT,
    "selected_group_split_trial": int(selected_trial),
    "mixed_label_hash_groups_train_validation": int(len(conflict_stats)),
    "rows_in_mixed_label_groups_train_validation": int(conflict_stats["rows"].sum()),
    "groups_with_hash_collision_on_exact_77_feature_comparison": int(conflict_stats["hash_collision_detected"].sum()),
    "groups_with_exactly_one_77_feature_vector": int((conflict_stats["exact_77_feature_vectors"] == 1).sum()),
    "largest_group": {
        "feature_hash": str(largest_hash), "rows": int(len(largest_group)),
        "labels": sorted(largest_group["Label"].astype(str).unique().tolist()),
        "exact_distinct_77_feature_vectors": largest_exact_vectors,
        "source_file_counts": {str(k): int(v) for k, v in largest_group["source_file"].value_counts().items()},
        "source_row_indexing": "zero-based within original CSV"
    },
    "negative_nonnegative_semantic_records_train_validation": int(len(negative_records)),
    "negative_feature_value_occurrences_train_validation": negative_feature_value_occurrences,
    "negative_unique_source_rows_train_validation": negative_unique_source_rows,
    "negative_features_detected": sorted(negative_records["feature"].unique().tolist()) if len(negative_records) else [],
    "test_predictions_performed": False, "test_metrics_computed": False,
    "training_performed": False, "labels_modified": False, "source_data_modified": False,
    "final_model_lock_modified": False, "checkpoint_modified": False,
    "pandas_version": pd.__version__, "numpy_version": np.__version__
}
(OUT_DIR / "audit_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
report = f"""# FL-06 v1.3 source-row audit — completed

- Verified source files: {len(verified_files)}/8
- Dataset fingerprint: `{DATASET_FINGERPRINT}`
- Split fingerprint: `{SPLIT_FINGERPRINT}`
- Reconstructed mixed-label Train/Validation groups: {len(conflict_stats)}
- Rows in those groups: {int(conflict_stats['rows'].sum())}
- Groups with more than one exact 77-feature vector (possible 64-bit hash collision): {int(conflict_stats['hash_collision_detected'].sum())}
- Largest group: {len(largest_group)} rows; {largest_exact_vectors} exact 77-feature vector(s)
- Negative feature-value occurrences in selected nonnegative-semantic features, Train/Validation only: {negative_feature_value_occurrences}
- Unique source rows affected by at least one such negative value: {negative_unique_source_rows}

## Safety
No model training, prediction, Test evaluation, label correction, row deletion, checkpoint write, lock modification, or source-data modification occurred. Raw source rows are loaded to reconstruct the pre-existing fixed split; no Test-specific analysis, export, prediction, metric, or tuning is performed.

## Files
- `source_conflict_groups_train_validation.csv`
- `source_conflict_records_77_features_train_validation.csv`
- `negative_values_summary_train_validation.csv`
- `negative_values_records_train_validation.csv`
- `audit_summary.json`
"""
(OUT_DIR / "AUDIT_REPORT.md").write_text(report, encoding="utf-8")

print("\nAUDIT COMPLETE")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nNew audit files:")
for p in sorted(OUT_DIR.iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")


Reconstructed mixed-label groups: 614
Rows in those groups: 6342
Largest group rows: 3362
Largest group exact distinct 77-feature vectors: 1
Largest group labels: ['BENIGN', 'DoS Hulk']
Largest group source files: {'Wednesday-workingHours.pcap_ISCX.csv': 3360, 'Tuesday-WorkingHours.pcap_ISCX.csv': 1, 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv': 1}

AUDIT COMPLETE
{
  "status": "completed",
  "scope": "Conflict and negative-value analysis restricted to Train and Validation; raw source rows are loaded to reproduce the fixed split, but no Test-specific analysis, export, prediction, metric, or tuning is performed",
  "repository_head": "c5129c433862f519b9a1dd306bff25b2af8a2c54",
  "verified_source_files": 8,
  "raw_rows": 2830743,
  "rows_removed": 2867,
  "clean_rows": 2827876,
  "feature_count": 77,
  "class_count": 15,
  "dataset_fingerprint": "831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58",
  "split_fingerprint": "b67f17d162df41840629b22fa258b7e843cf54541c

In [8]:
# Compare v2 outputs with the archived v1 outputs; v1 is read-only.
V1_DIR = AUDIT_DIR / "source_row_audit_v1"
comparison_files = [
    "source_conflict_groups_train_validation.csv",
    "source_conflict_records_77_features_train_validation.csv",
    "negative_values_summary_train_validation.csv",
    "negative_values_records_train_validation.csv",
]

def canonical_csv(path):
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    frame = frame.reindex(sorted(frame.columns), axis=1)
    if len(frame):
        frame = frame.sort_values(list(frame.columns), kind="mergesort").reset_index(drop=True)
    return frame

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

file_comparisons = {}
for name in comparison_files:
    old_path, new_path = V1_DIR / name, OUT_DIR / name
    if not old_path.is_file() or not new_path.is_file():
        file_comparisons[name] = {"status": "MISSING_FILE", "v1_exists": old_path.is_file(), "v2_exists": new_path.is_file(), "exact_content_match": False}
        continue
    old_sha, new_sha = file_sha256(old_path), file_sha256(new_path)
    exact = old_sha == new_sha
    if not exact:
        exact = canonical_csv(old_path).equals(canonical_csv(new_path))
    file_comparisons[name] = {"status": "PASS" if exact else "DIFFERENT", "v1_sha256": old_sha, "v2_sha256": new_sha, "exact_content_match": bool(exact)}

v1_summary = json.loads((V1_DIR / "audit_summary.json").read_text(encoding="utf-8"))
v2_summary = json.loads((OUT_DIR / "audit_summary.json").read_text(encoding="utf-8"))
summary_keys = [
    "verified_source_files", "raw_rows", "rows_removed", "clean_rows", "feature_count", "class_count",
    "dataset_fingerprint", "split_fingerprint", "selected_group_split_trial",
    "mixed_label_hash_groups_train_validation", "rows_in_mixed_label_groups_train_validation",
    "groups_with_hash_collision_on_exact_77_feature_comparison", "groups_with_exactly_one_77_feature_vector",
    "largest_group", "negative_nonnegative_semantic_records_train_validation",
    "negative_features_detected", "test_predictions_performed", "test_metrics_computed",
    "training_performed", "labels_modified", "source_data_modified", "final_model_lock_modified", "checkpoint_modified"
]
summary_comparisons = {key: {"v1": v1_summary.get(key), "v2": v2_summary.get(key), "match": v1_summary.get(key) == v2_summary.get(key)} for key in summary_keys}
comparison = {
    "status": "completed",
    "v1_output_directory": "audit/source_row_audit_v1",
    "v2_output_directory": "audit/source_row_audit_v2_reproducibility",
    "dataset_fingerprint_match": v1_summary.get("dataset_fingerprint") == v2_summary.get("dataset_fingerprint"),
    "split_fingerprint_match": v1_summary.get("split_fingerprint") == v2_summary.get("split_fingerprint"),
    "summary_comparisons": summary_comparisons,
    "csv_comparisons": file_comparisons,
    "all_summary_fields_match": all(x["match"] for x in summary_comparisons.values()),
    "all_csv_outputs_match": all(x.get("exact_content_match") is True for x in file_comparisons.values()),
}
comparison["all_comparisons_passed"] = bool(comparison["all_summary_fields_match"] and comparison["all_csv_outputs_match"])
(OUT_DIR / "audit_comparison.json").write_text(json.dumps(comparison, indent=2, ensure_ascii=False), encoding="utf-8")
with (OUT_DIR / "AUDIT_REPORT.md").open("a", encoding="utf-8") as report_file:
    report_file.write("\n## Reproducibility comparison against v1\n\n")
    report_file.write(f"- All summary fields match: {comparison['all_summary_fields_match']}\n")
    report_file.write(f"- All four CSV outputs match exactly (order-insensitive fallback): {comparison['all_csv_outputs_match']}\n")
    report_file.write(f"- Overall comparison passed: {comparison['all_comparisons_passed']}\n")
print(json.dumps(comparison, indent=2, ensure_ascii=False))
if not comparison["all_comparisons_passed"]:
    raise RuntimeError("v2 reproduction differs from v1. Preserve both outputs and investigate; do not overwrite or bypass.")

{
  "status": "completed",
  "v1_output_directory": "audit/source_row_audit_v1",
  "v2_output_directory": "audit/source_row_audit_v2_reproducibility",
  "dataset_fingerprint_match": true,
  "split_fingerprint_match": true,
  "summary_comparisons": {
    "verified_source_files": {
      "v1": 8,
      "v2": 8,
      "match": true
    },
    "raw_rows": {
      "v1": 2830743,
      "v2": 2830743,
      "match": true
    },
    "rows_removed": {
      "v1": 2867,
      "v2": 2867,
      "match": true
    },
    "clean_rows": {
      "v1": 2827876,
      "v2": 2827876,
      "match": true
    },
    "feature_count": {
      "v1": 77,
      "v2": 77,
      "match": true
    },
    "class_count": {
      "v1": 15,
      "v2": 15,
      "match": true
    },
    "dataset_fingerprint": {
      "v1": "831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58",
      "v2": "831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58",
      "match": true
    },
    "split_fingerprint"

## What to send back

When the notebook finishes, send me the printed `audit_summary.json` and `audit_comparison.json` outputs and attach these two files if possible:

1. `audit_comparison.json`
2. `source_file_verification.csv`
3. `source_conflict_groups_train_validation.csv`
4. `negative_values_summary_train_validation.csv`

The negative-value CSV is long-format (one row per feature/value occurrence); use `audit_summary.json` for the separate count of unique affected source rows. If possible, also attach the full `source_conflict_records_77_features_train_validation.csv`, I can inspect the largest mixed-label group and its exact feature profile without asking you to rerun anything.

**Important:** this notebook deliberately stops if the recreated dataset or split fingerprints differ from the committed v1.3 manifests. If that happens, send the error output; do not bypass the check. It does not push or commit anything to GitHub automatically.

## Automatic GitHub publication (run after the audit cell completes)

This optional final cell publishes only the eight allowlisted audit outputs. It requires a token with Contents: Read and Write; skip it if you only want to run the audit and download/attach the outputs. It refuses to proceed if the audit summary/fingerprints do not match, if the repository changed remotely since this runtime cloned it, or if any unexpected file would be staged. It never stages source data, checkpoints, model files, lock files, or Test artifacts.


In [9]:
# Publish the completed audit outputs to GitHub — allowlist only.
EXPECTED_DATASET = "831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58"
EXPECTED_SPLIT = "b67f17d162df41840629b22fa258b7e843cf54541ccfd6fff5910f325b70b812"
ALLOWED = {
    "AUDIT_REPORT.md",
    "audit_comparison.json",
    "source_file_verification.csv",
    "audit_summary.json",
    "negative_values_records_train_validation.csv",
    "negative_values_summary_train_validation.csv",
    "source_conflict_groups_train_validation.csv",
    "source_conflict_records_77_features_train_validation.csv",
}
if not OUT_DIR.is_dir():
    raise FileNotFoundError(f"Audit output folder not found: {OUT_DIR}. Run the audit cells first.")
summary_path = OUT_DIR / "audit_summary.json"
if not summary_path.is_file():
    raise FileNotFoundError("audit_summary.json is missing; refusing to publish.")
published_summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert published_summary.get("status") == "completed", "Audit is not marked completed."
assert published_summary.get("dataset_fingerprint") == EXPECTED_DATASET, "Dataset fingerprint mismatch."
assert published_summary.get("split_fingerprint") == EXPECTED_SPLIT, "Split fingerprint mismatch."
assert published_summary.get("test_predictions_performed") is False
assert published_summary.get("test_metrics_computed") is False
assert published_summary.get("training_performed") is False
actual_names = {p.name for p in OUT_DIR.iterdir() if p.is_file()}
if actual_names != ALLOWED:
    raise RuntimeError(f"Output allowlist mismatch. Expected {sorted(ALLOWED)}, found {sorted(actual_names)}")
for name in sorted(ALLOWED):
    p = OUT_DIR / name
    if p.stat().st_size > 10 * 1024 * 1024:
        raise RuntimeError(f"Refusing to publish unexpectedly large file: {name} ({p.stat().st_size} bytes)")
# Require a clean checkout and ensure the remote branch has not advanced.
# The six freshly generated audit outputs are expected to be untracked.
# Permit ONLY those exact paths; refuse every unrelated modification.
status_lines = git(["status", "--porcelain", "--untracked-files=all"], cwd=ROOT).stdout.splitlines()
expected_paths = {str((OUT_DIR / n).relative_to(ROOT)) for n in ALLOWED}
unexpected = []
for line in status_lines:
    if len(line) < 4:
        unexpected.append(line)
        continue
    code, path = line[:2], line[3:]
    if code == "??" and path in expected_paths:
        continue
    unexpected.append(line)
if unexpected:
    raise RuntimeError("Unexpected repository changes; refusing to publish:\\n" + "\\n".join(unexpected))
git(["fetch", "origin", "main"], cwd=ROOT)
local_head = git(["rev-parse", "HEAD"], cwd=ROOT).stdout.strip()
remote_head = git(["rev-parse", "origin/main"], cwd=ROOT).stdout.strip()
if local_head != remote_head:
    # Safe case: GitHub main only moved forward (e.g. this notebook's publisher
    # cell was added after the audit runtime cloned). Fast-forward the clean
    # checkout; refuse any divergence or local-only commits.
    ancestor = git(["merge-base", "--is-ancestor", "HEAD", "origin/main"], cwd=ROOT, check=False)
    if ancestor.returncode != 0:
        raise RuntimeError(f"GitHub main diverged from this runtime. Local={local_head}; remote={remote_head}. Nothing was staged or pushed.")
    git(["merge", "--ff-only", "origin/main"], cwd=ROOT)
git(["add", "--", str(OUT_DIR.relative_to(ROOT))], cwd=ROOT)
staged = set(git(["diff", "--cached", "--name-only"], cwd=ROOT).stdout.splitlines())
expected_paths = {str((OUT_DIR / n).relative_to(ROOT)) for n in ALLOWED}
if staged != expected_paths:
    raise RuntimeError(f"Staged paths differ from strict allowlist. Staged={sorted(staged)}")
git(["config", "user.name", "FL-06 Audit Bot"], cwd=ROOT)
git(["config", "user.email", "fl06-audit-bot@users.noreply.github.com"], cwd=ROOT)
git(["commit", "-m", "Add verified FL-06 source-row audit outputs"], cwd=ROOT)
new_commit = git(["rev-parse", "HEAD"], cwd=ROOT).stdout.strip()
git(["push", "origin", "HEAD:main"], cwd=ROOT)
git(["fetch", "origin", "main"], cwd=ROOT)
verified_remote = git(["rev-parse", "origin/main"], cwd=ROOT).stdout.strip()
if verified_remote != new_commit:
    raise RuntimeError(f"Push verification failed: local commit {new_commit}, remote main {verified_remote}")
print("GITHUB PUBLICATION VERIFIED")
print("Commit:", new_commit)
print("Destination:", f"{OUT_DIR.relative_to(ROOT)}/")
for name in sorted(ALLOWED):
    print(f"- {name} ({(OUT_DIR / name).stat().st_size:,} bytes)")
print("Only the six allowlisted audit outputs were committed. No source data, model, checkpoint, lock, or Test artifact was staged.")


GITHUB PUBLICATION VERIFIED
Commit: ce245f6f65610c305a9c8b770c3648e22635c131
Destination: CIC-IDS2017/FL-06_OneShot_GroupAware_v1_3_FinalPaper/audit/source_row_audit_v2_reproducibility/
- AUDIT_REPORT.md (1,402 bytes)
- audit_comparison.json (5,529 bytes)
- audit_summary.json (2,089 bytes)
- negative_values_records_train_validation.csv (309,890 bytes)
- negative_values_summary_train_validation.csv (8,249 bytes)
- source_conflict_groups_train_validation.csv (32,940 bytes)
- source_conflict_records_77_features_train_validation.csv (2,112,379 bytes)
- source_file_verification.csv (1,779 bytes)
Only the six allowlisted audit outputs were committed. No source data, model, checkpoint, lock, or Test artifact was staged.
